### 1. Definição e Setup
* **Objetivo:** Analisar quais fatores (estudo, frequência, educação dos pais) mais impactam o `final_score`.
* **Setup:** Importar `pandas`, `seaborn` e `plotly`.

### 2. Primeiro Contato (Auditoria)
* **Carregamento:** `df = pd.read_csv('student_performance.csv')`.
* **Inspeção:** Ver as colunas `parent_education`, `attendance_rate` e `study_hours_per_week`.
* **Qualidade:** Verificar se `attendance_rate` está entre 0-100 e se há nulos em `parent_education`.

### 3. Tratamento (Wrangling)
* **Limpeza:** Tratar valores "None" em `parent_education` (ex: substituir por "Unknown").
* **Conversão:** Garantir que `passed` (Yes/No) seja convertido para booleano ou numérico se necessário.
* **Engenharia:** Criar uma coluna de "Esforço Total" (estudo + frequência).

### 4. Exploração (EDA)
* **Distribuição:** Histograma do `final_score` para ver a curva de notas.
* **Relação:** Gráfico de dispersão entre `study_hours_per_week` e `final_score`.
* **Categorias:** Boxplot de `final_score` agrupado por `parent_education` ou `internet_access`.

### 5. Síntese e Insights
* **Visualização:** Heatmap de correlação entre as notas atuais e as anteriores (`previous_score`).
* **Conclusão:** Identificar se o acesso à internet ou atividades extracurriculares realmente garantem a aprovação (`passed`).

---

### ⏳ Onde vais gastar mais tempo neste dataset?
No **Tratamento (Passo 3)**.
* **O motivo:** Colunas como `parent_education` e `internet_access` são categóricas. Para fazeres correlações matemáticas, terás de as transformar ou agrupar, o que exige atenção para não enviesar os dados.


In [31]:
import pandas as pd 
import numpy as np
import plotly.express as px

df = pd.read_csv("student_performance.csv")

#### Visualização do head da base

In [32]:
df.head()

,student_id,gender,age,study_hours_per_week,attendance_rate,parent_education,internet_access,extracurricular,previous_score,final_score,passed
0,STU0001,Male,15,25,63.8,Bachelor,Yes,Yes,41,67,Yes
1,STU0002,Female,15,2,54.7,Bachelor,Yes,Yes,83,28,No
2,STU0003,Female,19,10,90.5,High School,Yes,No,73,49,No
3,STU0004,Male,16,26,66.8,High School,No,Yes,75,70,Yes
4,STU0005,Female,15,25,73.0,High School,No,Yes,67,77,Yes


#### Visualização do tail da base

In [33]:
df.tail()

,student_id,gender,age,study_hours_per_week,attendance_rate,parent_education,internet_access,extracurricular,previous_score,final_score,passed
495,STU0496,Female,19,6,78.3,Master,No,No,51,27,No
496,STU0497,Female,16,27,61.1,PhD,No,No,47,74,Yes
497,STU0498,Female,18,16,72.3,Master,No,Yes,52,61,Yes
498,STU0499,Male,17,29,91.3,NaN,Yes,No,39,86,Yes
499,STU0500,Male,15,29,75.4,High School,No,Yes,34,74,Yes


#### Metadados da base

In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   student_id            500 non-null    str    
 1   gender                500 non-null    str    
 2   age                   500 non-null    int64  
 3   study_hours_per_week  500 non-null    int64  
 4   attendance_rate       500 non-null    float64
 5   parent_education      383 non-null    str    
 6   internet_access       500 non-null    str    
 7   extracurricular       500 non-null    str    
 8   previous_score        500 non-null    int64  
 9   final_score           500 non-null    int64  
 10  passed                500 non-null    str    
dtypes: float64(1), int64(4), str(6)
memory usage: 43.1 KB


### Informações estatísticas da base

In [35]:
df.describe()

,age,study_hours_per_week,attendance_rate,previous_score,final_score
count,500.000000,500.000000,500.000000,500.000000,500.000000
mean,16.978000,15.312000,76.380600,62.986000,55.980000
std,1.434445,8.568167,13.817681,18.937451,15.373754
min,15.000000,2.000000,50.200000,30.000000,20.000000
25%,16.000000,8.000000,64.475000,46.000000,45.000000
50%,17.000000,15.000000,76.500000,64.000000,56.000000
75%,18.000000,23.000000,88.525000,79.000000,68.000000
max,19.000000,30.000000,100.000000,95.000000,95.000000


#### Visualização das colunas da base

In [36]:
df.columns

Index(['student_id', 'gender', 'age', 'study_hours_per_week',
       'attendance_rate', 'parent_education', 'internet_access',
       'extracurricular', 'previous_score', 'final_score', 'passed'],
      dtype='str')

#### Verificação de padrões dos dados (valores únicos)

In [37]:
colunas = list(df.columns)
def check_patter(df, colunas):
    for col in colunas:
        print(f'Coluna: {col} \n{ df[col].unique()}')
        print()
check_patter(df,colunas)

Coluna: student_id 
<StringArray>
['STU0001', 'STU0002', 'STU0003', 'STU0004', 'STU0005', 'STU0006', 'STU0007',
 'STU0008', 'STU0009', 'STU0010',
 ...
 'STU0491', 'STU0492', 'STU0493', 'STU0494', 'STU0495', 'STU0496', 'STU0497',
 'STU0498', 'STU0499', 'STU0500']
Length: 500, dtype: str

Coluna: gender 
<StringArray>
['Male', 'Female']
Length: 2, dtype: str

Coluna: age 
[15 19 16 18 17]

Coluna: study_hours_per_week 
[25  2 10 26  8 21  9 18  4 13 30 19 29 20 15  7 14 12 28  3 11 22 23 27
 24  6  5 16 17]

Coluna: attendance_rate 
[ 63.8  54.7  90.5  66.8  73.   85.2  72.7  81.7  84.2  95.7  57.   74.7
  69.2  63.3  93.8  88.1  99.3  74.3  93.6  75.9  61.5  66.5  73.6  70.6
  59.6  93.7  92.4  68.9  64.8  76.6  56.   83.5  96.5  56.6  77.2  55.4
  84.4  50.2  55.6  68.2  78.   59.   51.9  51.2  75.5  78.8  53.5  70.1
  78.5  82.7  54.2  60.   94.3  62.5  90.8  87.6  63.6  60.7  71.   69.5
  69.4  78.2  69.   97.6  82.5  73.7  80.4  83.9  88.6  75.3  94.6  96.
  94.1  74.4  75.8  85.6  

Coluna: previous_score 
[41 83 73 75 67 40 89 70 93 81 50 31 85 68 92 60 90 69 38 37 42 84 36 91
 66 80 77 43 86 48 61 64 88 65 44 30 57 52 49 58 46 47 55 62 39 74 82 45
 87 56 94 33 71 51 59 35 54 95 79 32 63 76 53 78 34 72]

Coluna: final_score 
[67 28 49 70 77 37 42 56 44 69 38 64 84 89 79 65 46 72 48 53 75 24 78 43
 41 62 31 27 55 82 23 57 61 59 68 80 66 71 76 34 52 74 58 40 45 60 50 54
 51 32 39 63 29 22 47 73 33 20 36 81 91 26 35 85 30 21 95 93 83 86]

Coluna: passed 
<StringArray>
['Yes', 'No']
Length: 2, dtype: str



#### Verificar se attendance_rate está entre 0-100

In [38]:
df.attendance_rate.min()

np.float64(50.2)

In [39]:
df.attendance_rate.max()

np.float64(100.0)

#### Verificação de valores nulos na base

In [40]:
df.isnull().sum()

student_id                0
gender                    0
age                       0
study_hours_per_week      0
attendance_rate           0
parent_education        117
internet_access           0
extracurricular           0
previous_score            0
final_score               0
passed                    0
dtype: int64

##### A coluna parent_education é a única com valores null

### Como não se sabe se os valores NaN ou null não foram digitados, assumirei que eles não existem

#### Neste caso vou prencher os valores NaN ou null com "Unknown"

In [41]:
df["parent_education"] = df["parent_education"].fillna("Unknown")

###### Checar se o preenchimento foi realizado com sucesso

In [42]:
df["parent_education"].unique()

<StringArray>
['Bachelor', 'High School', 'Master', 'Unknown', 'PhD']
Length: 5, dtype: str

#### Se os dados já estão como "Yes" ou "No", o objetivo é transformar essa informação de texto (string) para números (inteiros). Isso é essencial porque algoritmos matemáticos e gráficos de correlação não conseguem processar a palavra "Yes", mas conseguem processar o número "1"

In [43]:
# df["passed"] = np.where(df["passed"].replace({"Yes" : 1, "No" : 0}) == "Yes", "1", "0")
#df["passed"] = df["passed"].replace({"Yes" : 1, "No" : 0})
df["passed"] = df["passed"].map({"Yes" : 1, "No" : 0})

##### Conversão de para o tipo inteiro

In [44]:
df["passed"] = df["passed"].astype(int)

##### Checar o tipo

In [45]:
df["passed"].dtype

dtype('int64')

#### Engenharia: Criar uma coluna de "Esforço Total" (estudo + frequência).

In [46]:
df["Esforço Total"] = df["study_hours_per_week"] + df["attendance_rate"]

In [47]:
df.head()

,student_id,gender,age,study_hours_per_week,attendance_rate,parent_education,internet_access,extracurricular,previous_score,final_score,passed,Esforço Total
0,STU0001,Male,15,25,63.8,Bachelor,Yes,Yes,41,67,1,88.8
1,STU0002,Female,15,2,54.7,Bachelor,Yes,Yes,83,28,0,56.7
2,STU0003,Female,19,10,90.5,High School,Yes,No,73,49,0,100.5
3,STU0004,Male,16,26,66.8,High School,No,Yes,75,70,1,92.8
4,STU0005,Female,15,25,73.0,High School,No,Yes,67,77,1,98.0


In [48]:
df = df.rename(columns = {"Esforço Total" : "Total Effort"})

In [49]:
df.head()

,student_id,gender,age,study_hours_per_week,attendance_rate,parent_education,internet_access,extracurricular,previous_score,final_score,passed,Total Effort
0,STU0001,Male,15,25,63.8,Bachelor,Yes,Yes,41,67,1,88.8
1,STU0002,Female,15,2,54.7,Bachelor,Yes,Yes,83,28,0,56.7
2,STU0003,Female,19,10,90.5,High School,Yes,No,73,49,0,100.5
3,STU0004,Male,16,26,66.8,High School,No,Yes,75,70,1,92.8
4,STU0005,Female,15,25,73.0,High School,No,Yes,67,77,1,98.0


### Exploração (EDA)